# Multi-Task-MLP

Dieses Notebook implementiert und validiert den ersten tabellarischen Multi-Task-Klassifikator für `Crop` und `Stage`.

## Anforderungen erfassen

- Eingabe: vorverarbeitete Spektralbänder sowie AEZ und Month.
- Ausgabe: genau eine `Crop`- und eine `Stage`-Vorhersage je Zeile.
- Randfälle: unzulässige Zielspalten, nichtnumerische oder fehlende Merkmale und ungültige Sample Weights müssen abgewiesen werden.
- Erfolgskriterium: Der Klassifikator lässt sich mit `tune()` auf den Trainingsfolds suchen und liefert für `run()` einen DataFrame mit `Crop` und `Stage`.

## Projektstruktur einrichten

Das Modell, die Vorverarbeitung und der Experimentablauf liegen in `src/awp2/`. `tune()` und `run()` laden ausschließlich die gemeinsamen Split-Artefakte; das Notebook erzeugt keine eigenen Splits.

In [ ]:
from awp2.data import load_split
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import MultiTaskMLPClassifier
from awp2.preprocessing import PreprocessingConfig

split = load_split()
split.X_train.shape, split.y_train.shape

## Kernfunktion implementieren

`MultiTaskMLPClassifier` implementiert den gemeinsamen MLP-Encoder und die beiden Klassifikationsköpfe in `src/awp2/models/multi_task.py`. Die Losses sind je Zielklasse gewichtet; deshalb wird keine zusätzliche paarweise Sample-Gewichtung gesetzt.

In [ ]:
HIDDEN_UNITS = [32, 64]
HIDDEN_LAYERS = [1, 2]
DROPOUT = [0.2, 0.4]
PREPROCESSING = PreprocessingConfig(use_spectral_standard_scale=True)

model = MultiTaskMLPClassifier()
model.get_params(deep=True)

## Tests schreiben und ausführen

Die Unit-Tests prüfen Zielvertrag, Pipeline-Integration, Klonbarkeit und fehlerhafte Zielspalten. Sie müssen vor dem teuren CV-Lauf bestehen.

In [ ]:
import subprocess

from awp2.config import PROJECT_ROOT

completed = subprocess.run(
    ["uv", "run", "python", "-m", "unittest", "tests/test_multi_task.py"],
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)

## Ergebnisse prüfen

Die kleine Suche verändert nur Kapazität und Regularisierung des gemeinsamen MLP-Teils. Die Auswahl erfolgt ausschließlich über die mittlere CV-BAcc kombiniert auf den fünf Trainingsfolds.

In [ ]:
tuned = tune(
    model,
    TuneConfig(
        name="multi_task_mlp_search",
        approach="multi-task",
        description=(
            "Multi-Task-MLP mit gemeinsamem Encoder; kleine Suche über Kapazität und "
            "Regularisierung auf den Trainingsfolds."
        ),
        preprocessing=PREPROCESSING,
        param_grid={
            "hidden_units": HIDDEN_UNITS,
            "hidden_layers": HIDDEN_LAYERS,
            "dropout": DROPOUT,
        },
    ),
)
tuned.candidates

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="multi_task_mlp",
        approach="multi-task",
        description="CV-Gewinner der kleinen Suche für das tabellarische Multi-Task-MLP.",
        preprocessing=PREPROCESSING,
        tuning_run=tuned.run_id,
    ),
)
result.metrics

## Auswertung und Abbildungen

Die folgenden Abbildungen zeigen die CV-Streuung, die Fehler je Ziel und die häufigsten Verwechslungen. Sie werden in `docs/modelle/img/` gespeichert und auf dieser Ansatzseite eingebunden.

Nach dem einmaligen Holdout-Lauf werden die Metriken, die Run-ID und die wichtigsten Fehlerbilder in `docs/modelle/experimente.md` und `docs/modelle/ansaetze/multi-task.md` festgehalten. Ungültige Crop/Stage-Paare bleiben Teil der Auswertung; eine Paar-Beschränkung ist eine getrennte Folgehypothese.

In [ ]:
from IPython.display import display

from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import (
    plot_class_recall,
    plot_top_confusions,
    plot_tuning_candidates,
    save_doc_figure,
)

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="hidden_layers",
    validation_score=result.metrics.bacc_combined,
)
save_doc_figure(tuning_ax.figure, "multi_task_mlp_tuning", MODEL_DOCS_FIGURES_DIR)

figures = {
    "confusion": plot_confusion_matrices(result.y_val, result.y_pred),
    "recall": plot_class_recall(result.y_val, result.y_pred),
    "confusions": plot_top_confusions(result.y_val, result.y_pred),
}
for name, figure in figures.items():
    save_doc_figure(figure, f"multi_task_mlp_{name}", MODEL_DOCS_FIGURES_DIR)

display(tuning_ax.figure)
for figure in figures.values():
    display(figure)

## CV-only: L2-Regularisierung

Die Architektur aus der ersten Suche bleibt fest. Diese Suche prüft ausschließlich, ob L2-Regularisierung die Generalisierung des gemeinsamen MLP-Teils verbessert. Der gemeinsame Holdout wird dabei nicht erneut verwendet.

In [ ]:
WINNING_ARCHITECTURE = {
    "hidden_units": 64,
    "hidden_layers": 2,
    "dropout": 0.2,
}
WEIGHT_DECAY = [0.00001, 0.0001, 0.001]

regularization_tuned = tune(
    MultiTaskMLPClassifier(**WINNING_ARCHITECTURE),
    TuneConfig(
        name="multi_task_mlp_weight_decay_search",
        approach="multi-task",
        description="L2-Regularisierung bei der CV-gewählten Multi-Task-MLP-Architektur.",
        preprocessing=PREPROCESSING,
        param_grid={"weight_decay": WEIGHT_DECAY},
    ),
)
regularization_tuned.candidates

## CV-only: Preprocessing-Varianten

Jede Variante verändert genau einen Aspekt gegenüber den standardisierten Spektren mit AEZ und zyklischem Month. Die L2-gewählte Architektur bleibt fest; die drei CV-Scores sind damit der Repräsentation zuzuordnen und werden nicht auf dem Holdout verglichen.

In [ ]:
import pandas as pd

PREPROCESSING_VARIANTS = {
    "Vegetationsindizes": PreprocessingConfig(
        use_spectral_standard_scale=True,
        use_vegetation_indices=True,
    ),
    "PCA-20": PreprocessingConfig(
        use_spectral_standard_scale=True,
        spectral_pca_components=20,
    ),
    "Savitzky-Golay": PreprocessingConfig(
        use_spectral_standard_scale=True,
        use_savgol=True,
        savgol_window_length=7,
        savgol_polyorder=2,
    ),
}

preprocessing_tuned = {
    name: tune(
        regularization_tuned.best_model,
        TuneConfig(
            name=f"multi_task_mlp_{name.lower().replace('-', '_')}_search",
            approach="multi-task",
            description=f"CV-only: Multi-Task-MLP mit {name}.",
            preprocessing=preprocessing,
            param_grid={"weight_decay": [regularization_tuned.best_params["weight_decay"]]},
        ),
    )
    for name, preprocessing in PREPROCESSING_VARIANTS.items()
}

preprocessing_scores = pd.DataFrame(
    [
        {"Variante": "Standard + L2", "CV-BAcc kombiniert": regularization_tuned.best_score},
        *[
            {"Variante": name, "CV-BAcc kombiniert": tuned_variant.best_score}
            for name, tuned_variant in preprocessing_tuned.items()
        ],
    ]
).sort_values("CV-BAcc kombiniert", ascending=False)
preprocessing_scores